In [ ]:
import matplotlib.pyplot as plt
import math
from math import log
import pandas as pd
import numpy as np
import random

### Load data

#### MovieLens

In [ ]:
df_links = pd.read_csv('data/movielens/ml-latest-small/links.csv')
df_movies = pd.read_csv('data/movielens/ml-latest-small/movies.csv')
df_ratings = pd.read_csv('data/movielens/ml-latest-small/ratings.csv')
df_tags = pd.read_csv('data/movielens/ml-latest-small/tags.csv')

In [ ]:
df_tags.head()

#### Instacart

In [ ]:
#instacart
df_products = pd.read_csv('data/instacart/products.csv')
df_carts_prior = pd.read_csv('data/instacart/order_products__prior.csv')
df_carts_train = pd.read_csv('data/instacart/order_products__train.csv')
df_carts = pd.concat([df_carts_prior, df_carts_train])
df_carts.head()

In [ ]:
df_carts.head()

In [ ]:
# transforms the data into a long list of lists, each containing product IDs 
# that were purchased in the same order
carts = df_carts[['order_id', 'product_id']].groupby('order_id')['product_id'].apply(list).to_list()
carts[0:3]

### Ranking

Recommend 10 movies based on recent ratings

Use techniques to make sure the recommendation is reliable

In [ ]:
... code here ...

### Association rule mining

Calculate the number of frequent itemsets with varying levels for support

Try to guess what value of minimum support would be reasonable

Calculate association rules and find the one whose subsequent item has the least support (the one more in the tail)

#### Priori (Apyori)

In [ ]:
from apyori import apriori

In [ ]:
association_rules = apriori(carts, min_support=XXX, 
                            min_confidence=XXX,
                            min_lift=XXX, min_length=XXX)
association_rules = list(association_rules)

In [ ]:
association_rules

In [ ]:
idx = 5 #prints the 5th association rule

rule = association_rules[idx]
frequent_itemset = rule.items
support = rule.support

antecedent = rule.ordered_statistics[0].items_base
antecedent = [df_products.iloc[a-1]['product_name'] for a in antecedent]
consequent = rule.ordered_statistics[0].items_add
consequent = [df_products.iloc[c-1]['product_name'] for c in consequent]
lift = rule.ordered_statistics[0].lift
confidence = rule.ordered_statistics[0].confidence

print(f'{antecedent}->{consequent}')
print(f'support = {support}')
print(f'confidence = {confidence}')
print(f'lift = {lift}')

#### FP-growth (mlxtend)

In [ ]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpmax, fpgrowth
from mlxtend.frequent_patterns import association_rules

In [ ]:
# encode the dataset into a orders x items binary sparse matrix
te = TransactionEncoder()
te_data = te.fit(carts).transform(carts, sparse=True)
df = pd.DataFrame.sparse.from_spmatrix(te_data, columns=te.columns_)
# product indices must either start from 0 or be strings
df.columns = [str(i) for i in df.columns] 
# alternatively, reduce ids by 1
#carts_modified = [[carts[l][i]-1 for i in range(0, len(carts[l]))] for l in range(0, len(carts))]

In [ ]:
frequent_itemsets = fpgrowth(df, min_support=xxxx, use_colnames=True, verbose=1)

In [ ]:
rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=xxxx)
rules